In [32]:
import math
import random

In [33]:
def sigmoid(z):
    return 1.0 / (1 + math.exp(-z))

def cross_entropy_loss(predict, ground_truth):
    N = len(ground_truth)
    total_loss = 0.0
    for y_pred, y in zip(predict, ground_truth):
        for yp, yt in zip(y_pred, y):
            total_loss += yt * math.log(yp) + (1 - yt) * math.log(1 - yp)
    return - total_loss / N

In [34]:
class Neuron:
    def __init__(self, input_count, weights=None, bias=None):
        self.weights = weights if weights is not None else [random.uniform(-1, 1) for _ in range(input_count)]
        self.bias = bias if bias is not None else random.uniform(-1, 1)
        self.delta = 0.0
        self.output = 0.0
        self.inputs = []

    def activate(self, inputs, mode='train'):
        z = sum(w * x for w, x in zip(self.weights, inputs)) + self.bias
        output = sigmoid(z)
        if mode == 'train':
            self.inputs = inputs
            self.output = output
        return output

    def update(self, lr):
        for i in range(len(self.weights)):
            self.weights[i] -= lr*self.delta * self.inputs[i]
        self.bias -= lr * self.delta
    
class Layer:
    def __init__(self, neuron_count, input_count_per_neuron, weights=None, bias=None):
        self.neurons = [
            Neuron(
                input_count=input_count_per_neuron,
                weights=weights[i] if weights is not None else None,
                bias=bias[i] if bias is not None else None
            )
            for i in range(neuron_count)
        ]

    def forward(self, inputs, mode='train'):
        return [neuron.activate(inputs, mode) for neuron in self.neurons]

    def output(self):
        return [neuron.output for neuron in self.neurons]

    
class NeuralNetwork:
    def __init__(self, neuron_counts, weights=None, bias=None):
        self.layers = []
        for i in range(1, len(neuron_counts)):
            neuron_count = neuron_counts[i]
            input_count = neuron_counts[i - 1]
            layer_weights = weights[i - 1] if weights is not None else None
            layer_bias = bias[i - 1] if bias is not None else None
            layer = Layer(
                neuron_count=neuron_count,
                input_count_per_neuron=input_count,
                weights=layer_weights,
                bias=layer_bias
            )
            self.layers.append(layer)


    def feed_forward(self, inputs):
        for layer in self.layers:
            inputs = layer.forward(inputs, mode='train')
        return inputs
    
    def predict(self, inputs):
        for layer in self.layers:
            inputs = layer.forward(inputs, mode='eval')
        return inputs
    
    def backpropagation(self, ground_truth):
        output_layer = self.layers[-1]
        for neuron, target in zip(output_layer.neurons, ground_truth):
            neuron.delta = neuron.output - target

        for i in range(len(self.layers) - 2, -1, -1):
            current_layer = self.layers[i]
            next_layer = self.layers[i+1]
            for j in range(len(current_layer.neurons)):
                neuron = current_layer.neurons[j]
                grad = sum(next_neuron.delta * next_neuron.weights[j] for next_neuron in next_layer.neurons)
                neuron.delta = grad * neuron.output * (1-neuron.output)
    
    def update_weights(self, lr):
        for layer in self.layers:
            for neuron in layer.neurons:
                neuron.update(lr)
    
    def train(self, X, y, epochs=1000, lr=0.5, threshold=1e-6):
        losses = []
        prev_loss = None
        for epoch in range(1, epochs + 1):
            for x_sample, y_sample in zip(X, y):
                self.feed_forward(x_sample)
                self.backpropagation(y_sample)
                self.update_weights(lr)

            if epoch % 1000 == 0:
                loss, _ = self.evaluate(X, y)
                losses.append(loss)
                print(f"Epoch {epoch} | Loss: {loss:.6f}")

                if prev_loss is not None and abs(prev_loss - loss) < threshold:
                    print(f'Model converged at epoch {epoch}.')
                    break
                
                prev_loss = loss
        return losses  

    def evaluate(self, X, y):
        preds = [self.predict(x) for x in X]   
        loss = cross_entropy_loss(preds, y)   
        return loss, preds
                    

In [35]:
weights = [
    # hidden layer
    [
        [-1, -1],
        [1, 1]
    ],
    # output layer
    [
        [1, 1]
    ]
]

bias = [
    # hidden layer
    [1.5, -0.5],
    # output layer
    [-1.5]
]
nn = NeuralNetwork(neuron_counts=[2, 2, 1], weights=weights, bias=bias)

In [36]:
X_train = [
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
]

y_train = [[0], [1], [1], [0]]

nn.train(X_train, y_train, epochs=10000, threshold=1e-3)

for x in X_train:
    y_pred = nn.feed_forward(x)
    print(f"Input: {x} -> Output: {y_pred}")

Epoch 1000 | Loss: 0.010044
Epoch 2000 | Loss: 0.004686
Epoch 3000 | Loss: 0.003047
Epoch 4000 | Loss: 0.002255
Model converged at epoch 4000.
Input: [0, 0] -> Output: [0.0025862562233642757]
Input: [0, 1] -> Output: [0.9981235297627661]
Input: [1, 0] -> Output: [0.9981181204318]
Input: [1, 1] -> Output: [0.0026630454288128157]
